In [ ]:
"""
0_Capture_Images.ipynb

This notebook captures synchronized RGB and depth images using an Intel RealSense
depth camera mounted on a Raspberry Pi. Images are saved once per second with
matching timestamps.

Outputs:
    RGB images: PNG format
    Depth images: NumPy arrays (.npy)

Hardware used:
    - Intel RealSense depth camera
    - Raspberry Pi
    - GPIO LED indicator

The LED blinks after each successful capture.
"""

import os
import cv2
import numpy as np
import pyrealsense2 as rs
import time
from datetime import datetime
import threading
import gpiod

# Initialize GPIO (used to blink LED during capture)
chip = gpiod.Chip('gpiochip4')
led_line = chip.get_line(21)
led_line.request(consumer="LED", type=gpiod.LINE_REQ_DIR_OUT)

# Output folder where RGB and depth images will be stored
output_folder = "/home/tartcherry/depth_images_camera_right"
timestamp_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
os.makedirs(output_folder, exist_ok=True)

# Initialize RealSense depth filters
hdr_merge_filter = rs.hdr_merge()
threshold_filter = rs.threshold_filter()
depth_to_disparity = rs.disparity_transform(True)
spatial_filter = rs.spatial_filter()
temporal_filter = rs.temporal_filter()
hole_filling_filter = rs.hole_filling_filter()
disparity_to_depth = rs.disparity_transform(False)

# Set threshold filter for expected trunk distance range (meters)
threshold_filter.set_option(rs.option.min_distance, 0.5)
threshold_filter.set_option(rs.option.max_distance, 2.5)

def process_depth_frame(depth_frame):
    """
    Apply a sequence of RealSense depth filters to improve depth quality.
    """
    depth_frame = depth_to_disparity.process(depth_frame)
    depth_frame = hdr_merge_filter.process(depth_frame)
    depth_frame = threshold_filter.process(depth_frame)
    depth_frame = spatial_filter.process(depth_frame)
    depth_frame = temporal_filter.process(depth_frame)
    depth_frame = hole_filling_filter.process(depth_frame)
    depth_frame = disparity_to_depth.process(depth_frame)
    return depth_frame

def capture_frame(pipeline, frames, frame_type, output_folder, current_time):
    """
    Save either a color frame or depth frame to disk.
    """
    if frame_type == "color":
        frame = frames.get_color_frame()
        frame_data = np.asanyarray(frame.get_data())
        filename = os.path.join(output_folder, f"{current_time}.png")
        cv2.imwrite(filename, frame_data)
        return filename

    elif frame_type == "depth":
        frame = frames.get_depth_frame()
        frame = process_depth_frame(frame)
        frame_data = np.asanyarray(frame.get_data())
        filename = os.path.join(output_folder, f"{current_time}.npy")
        np.save(filename, frame_data)
        return filename

def blink_led():
    """
    Blink the LED once to indicate a successful capture.
    """
    led_line.set_value(1)
    time.sleep(0.2)
    led_line.set_value(0)
    time.sleep(0.2)

def save_intrinsics_info(pipeline_profile, output_folder):
    """
    Save camera intrinsic parameters and stream resolutions.
    """
    depth_stream = pipeline_profile.get_stream(rs.stream.depth)
    color_stream = pipeline_profile.get_stream(rs.stream.color)

    depth_intrinsics = depth_stream.as_video_stream_profile().get_intrinsics()
    color_intrinsics = color_stream.as_video_stream_profile()

    color_resolution = (color_intrinsics.width(), color_intrinsics.height())
    depth_resolution = (depth_intrinsics.width, depth_intrinsics.height)

    info_path = os.path.join(output_folder, "camera_info.txt")

    with open(info_path, 'w') as f:
        f.write("=== Intrinsics and Resolution Info ===\n")
        f.write(f"Timestamp: {datetime.now()}\n\n")

        f.write("Depth Stream:\n")
        f.write(f"  Resolution: {depth_resolution[0]} x {depth_resolution[1]}\n")
        f.write(f"  fx: {depth_intrinsics.fx}\n")
        f.write(f"  fy: {depth_intrinsics.fy}\n")
        f.write(f"  cx: {depth_intrinsics.ppx}\n")
        f.write(f"  cy: {depth_intrinsics.ppy}\n\n")

        f.write("Color Stream:\n")
        f.write(f"  Resolution: {color_resolution[0]} x {color_resolution[1]}\n")


def capture_rgbd_data(output_folder):
    """
    Main capture loop for collecting RGB and depth images once per second.
    """

    pipeline = rs.pipeline()
    config = rs.config()

    config.enable_stream(rs.stream.color, 1280, 720, rs.format.bgr8, 30)
    config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

    pipeline_profile = pipeline.start(config)

    align = rs.align(rs.stream.color)

    # Save camera intrinsics to file
    save_intrinsics_info(pipeline_profile, output_folder)

    # Set RealSense laser power to maximum
    device = pipeline_profile.get_device()
    depth_sensor = device.first_depth_sensor()

    if depth_sensor.supports(rs.option.laser_power):
        laser_power_range = depth_sensor.get_option_range(rs.option.laser_power)
        depth_sensor.set_option(rs.option.laser_power, laser_power_range.max)
        print(f"Set laser power to maximum: {laser_power_range.max}")

    # Wait until the next full minute before starting capture
    current_time = time.time()
    time_to_wait = 60 - (current_time % 60)

    print(f"Waiting {int(time_to_wait)} seconds to start at the next full minute...")
    led_line.set_value(1)

    start_wait = time.time()

    while time.time() - start_wait < time_to_wait:
        frames = pipeline.wait_for_frames()
        frames = align.process(frames)
        time.sleep(0.01)

    led_line.set_value(0)
    print("Starting capture loop...")

    try:

        last_second = -1

        while True:

            now = datetime.now()
            current_second = now.second

            # Trigger capture when the second changes
            if current_second != last_second:

                last_second = current_second

                frames = pipeline.wait_for_frames()
                frames = align.process(frames)

                image_timestamp = now.strftime("%Y-%m-%d-%H-%M-%S.%f")[:-5]

                color_thread = threading.Thread(
                    target=capture_frame,
                    args=(pipeline, frames, "color", output_folder, image_timestamp)
                )

                depth_thread = threading.Thread(
                    target=capture_frame,
                    args=(pipeline, frames, "depth", output_folder, image_timestamp)
                )

                color_thread.start()
                depth_thread.start()

                color_thread.join()
                depth_thread.join()

                print(f"RGB image saved: {image_timestamp}.png")
                print(f"Depth data saved: {image_timestamp}.npy")

                blink_led()

            time.sleep(0.01)

    except KeyboardInterrupt:

        print("Stopping capture...")

    finally:

        pipeline.stop()
        led_line.set_value(0)
        led_line.release()


# Start image capture
capture_rgbd_data(output_folder)